In [1]:
# Imports
!pip -q install "trl==0.8.1" "peft>=0.10.0" "bitsandbytes>=0.43.0" "accelerate>=0.28.0" "transformers>=4.40.0"

import os
import json
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
)
from peft import PeftModel
from trl import PPOConfig, PPOTrainer, AutoModelForCausalLMWithValueHead

In [2]:
# Configs
MODEL_NAME = "Qwen/Qwen2-7B-Instruct"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = False # Change this to False for LoRA, True for QLoRA
NUM_TRAIN_EPOCHS = 5

SYSTEM_PROMPT = "You are a weather forecasting assistant. You should only output in JSON format"
TRAIN_SPLIT = 0.8
FEATURE_COLUMNS = ['date', 'temp_max', 'temp_min', 'precipitation', 'wind', 'weather']
WEATHER_LABELS = ["drizzle", "rain", "sun", "snow", "fog"]
EXTERNAL_DATASET = "seattle-weather.csv"
WINDOW_SIZES = [7, 14, 21, 28]

SFT_ADAPTER_DIR = "sft-qlora-adapter" if LOAD_IN_4BIT else "sft-lora-adapter"
RM_ADAPTER_DIR = "weather_reward_model-adapter"
PPO_OUTPUT_DIR = "ppo-qlora-adapter" if LOAD_IN_4BIT else "ppo-lora-adapter"
TRAIN_JSONL_PATH = "/content/weather_data/seattle_weather_chat_train.jsonl"

assert os.path.isdir(SFT_ADAPTER_DIR), f"Missing SFT adapter dir: {SFT_ADAPTER_DIR}"
assert os.path.isdir(RM_ADAPTER_DIR), f"Missing reward adapter dir: {RM_ADAPTER_DIR}"

In [3]:
# Data setup
def load_sft_train_samples():
    if "train_samples" in globals() and train_samples:
        return train_samples

    rows = []
    with open(TRAIN_JSONL_PATH, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

sft_train_samples = load_sft_train_samples()

In [4]:
# Setup policy and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_policy_prompt(system_prompt, user_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        # Fallback if chat template is unavailable
        return f"System: {system_prompt}\n\nUser: {user_prompt}\n\nAssistant:"

ppo_rows = []
for sample in sft_train_samples:
    messages = sample["messages"]
    system_prompt = next(m["content"] for m in messages if m["role"] == "system")
    user_prompt = next(m["content"] for m in messages if m["role"] == "user")
    ppo_rows.append({
        "query": build_policy_prompt(system_prompt, user_prompt),   # what PPO policy sees
        "reward_prompt": user_prompt,                               # what reward model saw during RM training
    })

ppo_dataset = Dataset.from_list(ppo_rows)

def collator(batch):
    return {k: [row[k] for row in batch] for k in batch[0]}

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
# Setup datatype and quantization config and base policy
dtype = (
    torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16 if torch.cuda.is_available()
    else torch.float32
)

quantization_config = None
if LOAD_IN_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=dtype if dtype in (torch.float16, torch.bfloat16) else torch.float16,
        bnb_4bit_use_double_quant=True,
    )

base_policy = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    device_map="auto",
    dtype=None if LOAD_IN_4BIT else dtype,
    quantization_config=quantization_config,
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [6]:
# Create peft and value head for PPO
policy_peft = PeftModel.from_pretrained(
    base_policy,
    SFT_ADAPTER_DIR,
    is_trainable=True,
)

policy_model = AutoModelForCausalLMWithValueHead.from_pretrained(policy_peft)
policy_model.pretrained_model.config.use_cache = False

In [7]:
# Setup reward model
reward_base = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    trust_remote_code=True,
    device_map="auto",
    torch_dtype=dtype,
)

reward_base.config.pad_token_id = tokenizer.pad_token_id

reward_model = PeftModel.from_pretrained(reward_base, RM_ADAPTER_DIR)
reward_model.config.pad_token_id = tokenizer.pad_token_id

if hasattr(reward_model, "base_model") and hasattr(reward_model.base_model, "config"):
    reward_model.base_model.config.pad_token_id = tokenizer.pad_token_id

reward_model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen2-7B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): Qwen2ForSequenceClassification(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [8]:
# PPO Config for trainer
ppo_config = PPOConfig(
    model_name=MODEL_NAME,
    learning_rate=1e-6,
    batch_size=4,
    mini_batch_size=1,
    gradient_accumulation_steps=1,
    ppo_epochs=2,
    target_kl=0.1,
    seed=3407,
    log_with=None,
)

ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=policy_model,
    ref_model=None,  # PPOTrainer creates/uses a reference model internally
    tokenizer=tokenizer,
    dataset=ppo_dataset,
    data_collator=collator,
)

device = ppo_trainer.accelerator.device

generation_kwargs = {
    "max_new_tokens": 96,
    "do_sample": True,
    "top_p": 0.9,
    "temperature": 0.7,
    "pad_token_id": tokenizer.pad_token_id,
    "eos_token_id": tokenizer.eos_token_id,
}

In [9]:
from tqdm.auto import tqdm
import math

steps_per_epoch = math.ceil(len(ppo_dataset) / ppo_config.batch_size)
total_steps = NUM_TRAIN_EPOCHS * steps_per_epoch

progress_bar = tqdm(total=total_steps, desc="PPO training")

for epoch in range(NUM_TRAIN_EPOCHS):
    for step, batch in enumerate(ppo_trainer.dataloader):
        query_texts = batch["query"]
        reward_prompts = query_texts

        query_tensors = [
            tokenizer(
                q,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_SEQ_LENGTH,
            ).input_ids.squeeze(0).to(device)
            for q in query_texts
        ]

        response_tensors = ppo_trainer.generate(
            query_tensors,
            return_prompt=False,
            **generation_kwargs,
        )

        response_texts = [
            tokenizer.decode(r.squeeze(), skip_special_tokens=True).strip()
            for r in response_tensors
        ]

        reward_texts = [
            f"{prompt}\n{resp}" for prompt, resp in zip(reward_prompts, response_texts)
        ]

        reward_inputs = tokenizer(
            reward_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
        ).to(reward_model.device)

        with torch.no_grad():
            reward_scores = reward_model(**reward_inputs).logits.squeeze(-1).float()

        rewards = [score.detach() for score in reward_scores]

        stats = ppo_trainer.step(query_tensors, response_tensors, rewards)

        progress_bar.update(1)
        mean_reward = torch.stack(rewards).mean().item()
        progress_bar.set_postfix({
            "epoch": epoch + 1,
            "reward": f"{mean_reward:.4f}"
        })

progress_bar.close()

os.makedirs(PPO_OUTPUT_DIR, exist_ok=True)
ppo_trainer.model.pretrained_model.save_pretrained(PPO_OUTPUT_DIR)
tokenizer.save_pretrained(PPO_OUTPUT_DIR)

print(f"\nSaved PPO adapter to: {PPO_OUTPUT_DIR}")
print(f"Started from SFT adapter: {SFT_ADAPTER_DIR}")
print(f"Used reward adapter: {RM_ADAPTER_DIR}")

PPO training:   0%|          | 0/5775 [00:00<?, ?it/s]

You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


config.json:   0%|          | 0.00/766 [00:00<?, ?B/s]


Saved PPO adapter to: ppo-lora-adapter
Started from SFT adapter: sft-lora-adapter
Used reward adapter: weather_reward_model-adapter
